In [ ]:
# Fourth Coffee - carica i CSV in tabelle Delta gestite nel lakehouse di default del notebook.
# Prima: in Lakehouse > Files crea la cartella fourth_coffee e caricaci i CSV della cartella data/ del repository.
# Poi collega questo notebook al lakehouse come default ed eseguilo.
from pyspark.sql import functions as F

SRC = "Files/fourth_coffee"

TYPES = {
    "stores": {"OpenedOn": "ts"},
    "farms": {"AltitudeM": "int"},
    "certifications": {"ValidFrom": "ts", "ValidTo": "ts", "SuspendedFrom": "ts", "LastAuditDate": "ts"},
    "green_lots": {"PurchaseDate": "ts", "WeightKg": "double", "CuppingScore": "double", "MoisturePct": "double"},
    "green_lot_farms": {"SharePct": "int"},
    "roast_batches": {"RoastDate": "ts", "RoastWeightKg": "double"},
    "packaging_runs": {"PackagingDate": "ts", "BagsProduced": "int", "BestBefore": "ts"},
    "shipments": {"ship_date": "ts", "delivery_date": "ts", "bags_shipped": "int"},
    "equipment": {"install_date": "ts"},
    "machine_telemetry": {"timestamp": "ts", "pump_pressure_bar": "double", "boiler_temp_c": "double"},
    "complaints": {"ComplaintDate": "ts"},
}

for table, casts in TYPES.items():
    df = spark.read.option("header", True).csv(f"{SRC}/{table}.csv")
    for col, typ in casts.items():
        if typ == "ts":
            df = df.withColumn(col, F.when(F.col(col) == "", None).otherwise(F.to_timestamp(col)))
        else:
            df = df.withColumn(col, F.col(col).cast(typ))
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(table)
    print(table, df.count())
